In [1]:
from pycromanager import Core

core = Core()
print(core)

C:\Users\Cell Culture Scope\AppData\Roaming\Python\Python311\site-packages\pyjavaz\bridge.py:347: UserWarning: Version mistmatch between Java ZMQ server and Python client. 
Java ZMQ server version: 4.0.0
Python client expected version: 5.1.0
 To fix, update to BOTH Java and Python sides of bridge
  warnings.warn(


In [1]:
import numpy as np

from pycromanager import Acquisition, Core, Studio, multi_d_acquisition_events

mmc = Core()
mmStudio = Studio()

In [5]:
# Data set parameters
path = r"C:\test"
name = "pycromanager_test"
total_duration = 40  # in seconds
subset_interval = 20  # in seconds

# z stack parameters
z_start = -2.5
z_end = 2.5
z_step = 0.25
relative = True
sequence = False

# time series parameters
duration = 2  # in seconds
framerate = 10

num_subsets = np.ceil(total_duration / subset_interval).astype(int)
num_time_points = duration * framerate
z_sequence = np.arange(z_start, z_end + z_step, z_step)
num_z_slices = len(z_sequence)

In [3]:
from pymmcore_plus import CMMCorePlus
from useq import MDAEvent

# Create the core instance.
mmc = CMMCorePlus.instance()  
mmc.loadSystemConfiguration()  

# Create a super-simple sequence, with one event
mda_sequence = [MDAEvent()] 
mmc.getLoadedDevices()
# Run it!
#mmc.run_mda(mda_sequence)

('DHub',
 'Camera',
 'Dichroic',
 'Emission',
 'Excitation',
 'Objective',
 'Z',
 'Path',
 'XY',
 'White Light Shutter',
 'Autofocus',
 'LED',
 'LED Shutter',
 'Core')

In [6]:
# setup cameras -- this property may change depending on the particular camera used
mmc.set_property("Andor", "Exposure", framerate)
mmc.set_property("TIDiaLamp", "Intensity", 3)
# setup z stage
z_stage = mmc.get_focus_device()

if relative:
    z_pos = mmc.get_position(z_stage)

    z_sequence += z_pos

if sequence:
    mmc.set_property(z_stage, "UseSequence", "Yes")

In [7]:
events = []
for s in range(num_subsets):
    for t in range(num_time_points):
        events.append(
            {
                "axes": {"subset": s, "time": t, "z": 0},
                "z": z_sequence[0],
                "min_start_time": s * subset_interval,
            }
        )
    for z in range(num_z_slices):
        events.append(
            {
                "axes": {"subset": s, "time": num_time_points, "z": z},
                "z": z_sequence[z],
                "min_start_time": s * subset_interval,
            }
        )
print(events)

[{'axes': {'subset': 0, 'time': 0, 'z': 0}, 'z': 501.4500075094402, 'min_start_time': 0}, {'axes': {'subset': 0, 'time': 1, 'z': 0}, 'z': 501.4500075094402, 'min_start_time': 0}, {'axes': {'subset': 0, 'time': 2, 'z': 0}, 'z': 501.4500075094402, 'min_start_time': 0}, {'axes': {'subset': 0, 'time': 3, 'z': 0}, 'z': 501.4500075094402, 'min_start_time': 0}, {'axes': {'subset': 0, 'time': 4, 'z': 0}, 'z': 501.4500075094402, 'min_start_time': 0}, {'axes': {'subset': 0, 'time': 5, 'z': 0}, 'z': 501.4500075094402, 'min_start_time': 0}, {'axes': {'subset': 0, 'time': 6, 'z': 0}, 'z': 501.4500075094402, 'min_start_time': 0}, {'axes': {'subset': 0, 'time': 7, 'z': 0}, 'z': 501.4500075094402, 'min_start_time': 0}, {'axes': {'subset': 0, 'time': 8, 'z': 0}, 'z': 501.4500075094402, 'min_start_time': 0}, {'axes': {'subset': 0, 'time': 9, 'z': 0}, 'z': 501.4500075094402, 'min_start_time': 0}, {'axes': {'subset': 0, 'time': 10, 'z': 0}, 'z': 501.4500075094402, 'min_start_time': 0}, {'axes': {'subset':

In [8]:
with Acquisition(directory=path, name=name) as acq:
    acq.acquire(events)